In [1]:
!pip install -q --upgrade google-generativeai python-dotenv


In [2]:
import os, time, sys
import google.generativeai as genai
from dotenv import load_dotenv


In [3]:
load_dotenv("keys.env")          # or load_dotenv() if your file is .env
assert os.getenv("GEMINI_API_KEY"), "Missing GEMINI_API_KEY in keys.env/.env"
genai.configure(api_key=os.environ["GEMINI_API_KEY"])


In [4]:
MODEL_NAME = "gemini-2.5-flash"

generation_config = {
    "temperature": 0.0,
    "max_output_tokens": 128,
    "candidate_count": 1,
}

# keep safety defaults for now (less chance of empty blocks)
model = genai.GenerativeModel(
    model_name=MODEL_NAME,
    generation_config=generation_config,
)


In [5]:
def ask_once(prompt: str) -> str:
    """One-shot, non-streaming (most reliable place to start)."""
    try:
        resp = model.generate_content(prompt)
        # Combine any text parts that came back
        return "".join([p.text for p in getattr(resp, "candidates", [])[0].content.parts])
    except Exception as e:
        return f"[error] {e}"

print(ask_once("Say hi in one short sentence."))


Hi!


In [6]:
def ask_stream(prompt: str, debug=False):
    """
    Stream tokens; print TTFT and total time.
    If no text chunks arrive, we fall back to non-streaming once.
    """
    print(f"\nYou: {prompt}")
    print("Model:", end=" ", flush=True)

    start = time.time()
    ttft = None
    got_text = False
    full = []

    try:
        stream = model.generate_content(prompt, stream=True)
        for chunk in stream:
            # Some chunks may not include text (control/safety). Use .text when present.
            text = getattr(chunk, "text", None)
            if debug and text is None:
                # Show the chunk object if needed for debugging
                print(f"\n[debug chunk] {chunk!r}")
            if not text:
                continue

            if ttft is None:
                ttft = time.time() - start
                print(f"\nTTFT: {ttft:.3f}s\n", flush=True)

            got_text = True
            full.append(text)
            print(text, end="", flush=True)

    except Exception as e:
        print(f"\n[stream error] {e}", file=sys.stderr)

    print()
    total = time.time() - start

    if got_text:
        print("\n" + "-"*48)
        print(f"TTFT: {ttft:.3f}s   Total: {total:.3f}s   Chars: {len(''.join(full))}")
    else:
        # Fallback to non-streaming if stream yielded no visible text
        print("[no streamed text, trying fallback…]")
        print(ask_once(prompt))

# try it:
ask_stream("Give me two concise tips to reduce LLM response latency.")



You: Give me two concise tips to reduce LLM response latency.
Model: 


[stream error] 



[no streamed text, trying fallback…]



In [7]:
print(ask_once("hi"))
ask_stream("hi")




You: hi
Model: 


[stream error] 



[no streamed text, trying fallback…]



In [8]:
print(ask_once("Give me two concise tips to reduce LLM response latency."))
ask_stream("Give me two concise tips to reduce LLM response latency.")




You: Give me two concise tips to reduce LLM response latency.
Model: 


[stream error] 



[no streamed text, trying fallback…]
[error] 429 You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. [violations {
  quota_metric: "generativelanguage.googleapis.com/generate_content_free_tier_requests"
  quota_id: "GenerateRequestsPerMinutePerProjectPerModel-FreeTier"
  quota_dimensions {
    key: "model"
    value: "gemini-2.5-flash"
  }
  quota_dimensions {
    key: "location"
    value: "global"
  }
  quota_value: 10
}
, links {
  description: "Learn more about Gemini API quotas"
  url: "https://ai.google.dev/gemini-api/docs/rate-limits"
}
, retry_delay {
  seconds: 54
}
]
